In [1]:
import torch

In [2]:
import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

from akita_model.model import SeqNN

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
model = SeqNN()
model.load_state_dict(torch.load("/home1/smaruj/pytorch_akita/models/finetuned/mouse/Hsieh2019_mESC/checkpoints/Akita_v2_mouse_Hsieh2019_mESC_model0_finetuned.pth", map_location=device))
model.eval()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 128, kernel_size=(15,), stride=(1,), padding=(7,), bias=False)
    (batch_norm): BatchNorm1d(128, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(128, 128, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, paddi

In [5]:
import pandas as pd

In [7]:
df = pd.read_csv(f"/scratch1/smaruj/generate_genomic_boundary/unsuccessful_all_folds_-0.5.tsv", sep="\t")

In [8]:
boundary_mask_path = "/scratch1/smaruj/generate_genomic_boundary/boundary_indices.pt"

In [9]:
import sys
sys.path.insert(0, "/home1/smaruj/ledidi")
from ledidi import Ledidi

In [10]:
bin_size = 2048
cropping_applied = 64
padding_bins = 2
padding = padding_bins * bin_size

slice_0_bins = [256]
slice_0_start = (min(slice_0_bins) + cropping_applied - padding_bins) * bin_size
slice_0_end = (max(slice_0_bins) + 1 + cropping_applied + padding_bins) * bin_size

In [13]:
folds = list(df["fold"].unique())

In [14]:
folds

[0, 1, 2, 3, 4, 5, 6, 7]

In [ ]:
for row in df.itertuples(index=False):
    chrom, pred_start, pred_end = row.chrom, row.centered_start, row.centered_end
    FOLD = row.fold
    
    print(f"Boundary generation for genome location: {chrom}:{pred_start}-{pred_end}")
    
    X = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/ohe_X/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_X.pt", weights_only=True, map_location=device)
    target = torch.load(f"/scratch1/smaruj/generate_genomic_boundary/targets/target_-0.2/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_target.pt", weights_only=True, map_location=device)
    tower_output_path = f"/scratch1/smaruj/generate_genomic_boundary/tower_outputs/fold{FOLD}/{chrom}_{pred_start}_{pred_end}_tower_out.pt"
    
    wrapper = Ledidi(model, 
                 input_loss=torch.nn.L1Loss(reduction='sum'), 
                 output_loss=torch.nn.L1Loss(reduction='sum'),
                 batch_size=1,
                 l=10.0,
                 max_iter=3000,
                 early_stopping_iter=2000,
                 return_history=True,
                 verbose=True,
                 bin_size=2048,
                 input_mask_slices_0=[256], # mid-bin
                 cropping_applied=64,
                 output_mask_path=boundary_mask_path,
                 use_semifreddo=True,
                 semifreddo_temp_output_path=tower_output_path,
                 punish_ctcf=False,
                 ctcf_meme_path=None
                 ).cuda()
    
    slice_0_torch = X[:, :, slice_0_start:slice_0_end]
    
    x_bar_slice_0, history = wrapper.fit_transform(X=slice_0_torch, y_bar=target)
    
    
    # torch.save(x_bar_slice_0[:,:,padding:-padding], f"/scratch1/smaruj/genomic_insertion_loci/results/{chrom}_{pred_start}_{pred_end}_slice.pt")
    

In [ ]:
for key in history:
    print(key)

In [ ]:
edit_positions = history["edit_positions"][:2751]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Convert your list of 2763 lists (each of length 2048) to a 2D array
edit_array = np.array(edit_positions)  # shape: (2763, 2048)

plt.figure(figsize=(15, 6))
plt.imshow(edit_array, aspect='auto', cmap='Greys', interpolation='none')

plt.xlabel('Sequence Position')
plt.ylabel('Iteration')
plt.title('Edit Positions Over Optimization Steps')
plt.colorbar(label='Edit (1) / No Edit (0)')

plt.tight_layout()
plt.show()

In [ ]:
from pyfaidx import Fasta

In [ ]:
fasta_file = "/project/fudenber_735/genomes/mm10/mm10.fa"
genome = Fasta(fasta_file)

In [ ]:
CTCF_PWM = "/home1/smaruj/IterativeMutagenesis/MA0139.1.meme"

In [ ]:
def read_meme_pwm_as_numpy(filename):
    pwm_list = []  # List to store PWM rows
    
    with open(filename, 'r') as file:
        in_matrix_section = False
        
        for line in file:
            line = line.strip()
            
            # Check if we are reading the PWM matrix
            if line.startswith("letter-probability matrix"):
                in_matrix_section = True  # Start reading matrix data
                continue  # Skip this header line
            
            # If we are in the matrix section, process the rows
            if in_matrix_section and line:
                pwm_row = [float(value) for value in line.split()]  # Parse values
                pwm_list.append(pwm_row)  # Append to the PWM list
            
            # If we encounter a new MOTIF or the end of file, stop matrix reading
            if line.startswith("MOTIF") and in_matrix_section:
                break
    
    # Convert the list to a numpy array
    pwm_array = np.array(pwm_list)
    
    return pwm_array

In [ ]:
pwm_CTCF = read_meme_pwm_as_numpy(CTCF_PWM)
pwm_CTCF_tensor = torch.from_numpy(pwm_CTCF.T).float()
motifs_dict = {"CTCF": pwm_CTCF_tensor}

In [ ]:
from tangermeme.tools import fimo

In [ ]:
hits = fimo.fimo(
    motifs=motifs_dict,
    sequences=x_bar_slice_0[:, :, 4096:-4096],
    threshold=1e-4,
    reverse_complement=True
)[0]

In [ ]:
hits

In [ ]:
edit_array = np.array(edit_positions)  # shape: (2763, 2048)

plt.figure(figsize=(15, 6))
plt.imshow(edit_array, aspect='auto', cmap='Greys', interpolation='none')

# Plot vertical shaded regions for CTCF motif positions
for _, row in hits.iterrows():
    start = row['start']
    end = row['end']
    plt.axvspan(start, end, color='red', alpha=0.3)

plt.xlabel('Sequence Position')
plt.ylabel('Iteration')
plt.title('Edit Positions Over Time with CTCF Motifs')
plt.colorbar(label='Edit (1) / No Edit (0)')

plt.tight_layout()
plt.show()